# ACDADA — Notebook 03: Anomaly Detection Agent

**Unsupervised / Semi-supervised Anomaly Detection**

This notebook implements:
1. Autoencoder-based anomaly detection (reconstruction error)
2. Variational Autoencoder (VAE) for probabilistic anomaly scoring
3. Isolation Forest (tree-based unsupervised)
4. Ensemble anomaly scorer combining all methods
5. Threshold calibration and evaluation
6. Model export for deployment

In [ ]:
# ============================================================
# DATASET LINKS (processed data from Notebook 01)
# ============================================================
#
# Original raw datasets:
# 1. CIC-IDS-2017: https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset
# 2. UNSW-NB15:    https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15
# 3. Bot-IoT:      https://www.kaggle.com/datasets/vigneshvenkateswaran/bot-iot-dataset
# 4. BETH Dataset:  https://www.kaggle.com/datasets/katehighnam/beth-dataset
#
# Processed data expected at: ../data/processed/<dataset>/
# Run Notebook 01 first to generate processed splits.
# ============================================================

## 1. Imports & Configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, average_precision_score, roc_curve,
    f1_score, accuracy_score
)
from sklearn.preprocessing import MinMaxScaler

import joblib
import gc

warnings.filterwarnings('ignore')

# Paths
PROCESSED_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
LOGS_DIR = Path('../logs')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')

# Hyperparameters
RANDOM_STATE = 42
BATCH_SIZE = 512
AE_EPOCHS = 100
AE_LR = 1e-3
AE_PATIENCE = 10

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## 2. Load Processed Data

In [ ]:
def load_dataset_splits(dataset_name: str, processed_dir: Path) -> dict:
    """Load preprocessed train/val/test splits."""
    dataset_dir = processed_dir / dataset_name
    if not dataset_dir.exists():
        print(f'[WARNING] {dataset_dir} does not exist.')
        return None
    data = {}
    for key in ['X_train', 'X_val', 'X_test',
                'y_train_binary', 'y_val_binary', 'y_test_binary']:
        filepath = dataset_dir / f'{key}.npy'
        if filepath.exists():
            data[key] = np.load(filepath)
    return data

# Load dataset
DATASET_NAME = 'cicids2017'
for name in ['unified', 'cicids2017', 'unsw_nb15', 'bot_iot']:
    if (PROCESSED_DIR / name).exists():
        DATASET_NAME = name
        break

print(f'Loading: {DATASET_NAME}')
data = load_dataset_splits(DATASET_NAME, PROCESSED_DIR)
if data is None:
    raise FileNotFoundError('Run Notebook 01 first.')

X_train = data['X_train']
X_val = data['X_val']
X_test = data['X_test']
y_train = data['y_train_binary']
y_val = data['y_val_binary']
y_test = data['y_test_binary']

N_FEATURES = X_train.shape[1]
print(f'Features: {N_FEATURES}')
print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
print(f'Train attack ratio: {y_train.mean():.3f}')

## 3. Prepare Anomaly Detection Training Data

For anomaly detection, we train primarily on **benign** traffic.
The model learns what "normal" looks like, then flags deviations.

In [ ]:
# Extract benign-only training data for the autoencoder
benign_mask_train = y_train == 0
X_train_benign = X_train[benign_mask_train]
print(f'Benign training samples: {X_train_benign.shape[0]:,} / {X_train.shape[0]:,}')

# For validation, keep both classes to evaluate detection
# Convert to tensors
train_tensor = torch.FloatTensor(X_train_benign)
val_tensor = torch.FloatTensor(X_val)
test_tensor = torch.FloatTensor(X_test)

train_dataset = TensorDataset(train_tensor, train_tensor)  # autoencoder: input=target
val_dataset = TensorDataset(val_tensor, torch.LongTensor(y_val))
test_dataset = TensorDataset(test_tensor, torch.LongTensor(y_test))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
                        num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
                         num_workers=0, pin_memory=True)

print(f'Train loader batches: {len(train_loader)}')
print(f'Val loader batches: {len(val_loader)}')

---
## 4. Autoencoder Model

Deep autoencoder trained to reconstruct benign traffic.
High reconstruction error → anomalous (likely attack).

In [ ]:
class DeepAutoencoder(nn.Module):
    """
    Deep Autoencoder for anomaly detection.
    
    Architecture:
    Encoder: input → 128 → 64 → 32 → latent_dim
    Decoder: latent_dim → 32 → 64 → 128 → input
    
    Anomaly score = MSE reconstruction error
    """
    
    def __init__(self, n_features: int, latent_dim: int = 16, dropout: float = 0.2):
        super().__init__()
        self.n_features = n_features
        self.latent_dim = latent_dim
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            
            nn.Linear(32, latent_dim),
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            
            nn.Linear(128, n_features),
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.constant_(m.bias, 0)
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        z = self.encode(x)
        x_recon = self.decode(z)
        return x_recon
    
    def anomaly_score(self, x):
        """Compute per-sample reconstruction error (MSE)."""
        x_recon = self.forward(x)
        mse = torch.mean((x - x_recon) ** 2, dim=1)
        return mse

# Test
ae_model = DeepAutoencoder(N_FEATURES, latent_dim=16).to(DEVICE)
dummy = torch.randn(4, N_FEATURES).to(DEVICE)
recon = ae_model(dummy)
scores = ae_model.anomaly_score(dummy)
print(f'Autoencoder: input={dummy.shape}, recon={recon.shape}, scores={scores.shape}')
print(f'Parameters: {sum(p.numel() for p in ae_model.parameters()):,}')

## 5. Variational Autoencoder (VAE)

In [ ]:
class VAEAnomalyDetector(nn.Module):
    """
    Variational Autoencoder for probabilistic anomaly detection.
    Anomaly score = reconstruction error + KL divergence.
    """
    
    def __init__(self, n_features: int, latent_dim: int = 16, dropout: float = 0.2):
        super().__init__()
        self.n_features = n_features
        self.latent_dim = latent_dim
        
        # Encoder
        self.encoder_shared = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
        )
        self.fc_mu = nn.Linear(64, latent_dim)
        self.fc_logvar = nn.Linear(64, latent_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Linear(128, n_features),
        )
    
    def encode(self, x):
        h = self.encoder_shared(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar
    
    def anomaly_score(self, x):
        """ELBO-based anomaly score."""
        x_recon, mu, logvar = self.forward(x)
        # Reconstruction error
        recon_error = torch.mean((x - x_recon) ** 2, dim=1)
        # KL divergence
        kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
        return recon_error + 0.1 * kl_div  # weighted combination

# Test
vae_model = VAEAnomalyDetector(N_FEATURES, latent_dim=16).to(DEVICE)
recon, mu, logvar = vae_model(dummy)
scores = vae_model.anomaly_score(dummy)
print(f'VAE: input={dummy.shape}, recon={recon.shape}, mu={mu.shape}, scores={scores.shape}')
print(f'Parameters: {sum(p.numel() for p in vae_model.parameters()):,}')

---
## 6. Training Engine for Autoencoders

In [ ]:
def vae_loss_fn(x, x_recon, mu, logvar, beta=1.0):
    """VAE loss = reconstruction + beta * KL divergence."""
    recon_loss = nn.functional.mse_loss(x_recon, x, reduction='mean')
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl_loss, recon_loss.item(), kl_loss.item()


class AnomalyDetectionTrainer:
    """
    Training engine for autoencoder-based anomaly detection.
    Supports both standard AE and VAE.
    """
    
    def __init__(self, model, lr=1e-3, device='cpu', is_vae=False):
        self.model = model.to(device)
        self.device = device
        self.is_vae = is_vae
        
        self.optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', patience=5, factor=0.5, min_lr=1e-6
        )
        self.criterion = nn.MSELoss()
        
        self.history = {'train_loss': [], 'val_loss': [], 'lr': []}
    
    def train_epoch(self, train_loader):
        self.model.train()
        total_loss = 0
        n_samples = 0
        
        for X_batch, _ in train_loader:
            X_batch = X_batch.to(self.device)
            self.optimizer.zero_grad()
            
            if self.is_vae:
                x_recon, mu, logvar = self.model(X_batch)
                loss, _, _ = vae_loss_fn(X_batch, x_recon, mu, logvar, beta=0.5)
            else:
                x_recon = self.model(X_batch)
                loss = self.criterion(x_recon, X_batch)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            
            total_loss += loss.item() * X_batch.size(0)
            n_samples += X_batch.size(0)
        
        return total_loss / n_samples
    
    @torch.no_grad()
    def compute_anomaly_scores(self, data_loader):
        """Compute anomaly scores for all samples."""
        self.model.eval()
        all_scores = []
        all_labels = []
        
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(self.device)
            scores = self.model.anomaly_score(X_batch)
            all_scores.extend(scores.cpu().numpy())
            all_labels.extend(y_batch.numpy())
        
        return np.array(all_scores), np.array(all_labels)
    
    @torch.no_grad()
    def validate_loss(self, val_loader):
        """Compute validation loss (reconstruction only, on all data)."""
        self.model.eval()
        total_loss = 0
        n_samples = 0
        
        for X_batch, _ in val_loader:
            X_batch = X_batch.to(self.device)
            if self.is_vae:
                x_recon, mu, logvar = self.model(X_batch)
                loss, _, _ = vae_loss_fn(X_batch, x_recon, mu, logvar, beta=0.5)
            else:
                x_recon = self.model(X_batch)
                loss = self.criterion(x_recon, X_batch)
            total_loss += loss.item() * X_batch.size(0)
            n_samples += X_batch.size(0)
        
        return total_loss / n_samples
    
    def fit(self, train_loader, val_loader, epochs=100, patience=10):
        """Train with early stopping."""
        model_name = self.model.__class__.__name__
        print(f'\nTraining {model_name}...')
        print(f'  Epochs: {epochs} | Patience: {patience}')
        print('-' * 60)
        
        best_val_loss = float('inf')
        best_state = None
        patience_counter = 0
        
        for epoch in range(1, epochs + 1):
            train_loss = self.train_epoch(train_loader)
            val_loss = self.validate_loss(val_loader)
            
            self.scheduler.step(val_loss)
            lr = self.optimizer.param_groups[0]['lr']
            
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['lr'].append(lr)
            
            if epoch % 10 == 0 or epoch == 1:
                print(f'  Epoch {epoch:3d}/{epochs} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | LR: {lr:.2e}')
            
            if val_loss < best_val_loss - 1e-6:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f'  Early stopping at epoch {epoch}')
                    break
        
        if best_state is not None:
            self.model.load_state_dict(best_state)
            print(f'  Restored best model (val_loss: {best_val_loss:.6f})')
        
        return self.history

---
## 7. Train Autoencoder

In [ ]:
# Standard Autoencoder
ae_model = DeepAutoencoder(N_FEATURES, latent_dim=16, dropout=0.2).to(DEVICE)
ae_trainer = AnomalyDetectionTrainer(ae_model, lr=AE_LR, device=DEVICE, is_vae=False)
ae_history = ae_trainer.fit(train_loader, val_loader, epochs=AE_EPOCHS, patience=AE_PATIENCE)

In [ ]:
# VAE
vae_model = VAEAnomalyDetector(N_FEATURES, latent_dim=16, dropout=0.2).to(DEVICE)
vae_trainer = AnomalyDetectionTrainer(vae_model, lr=AE_LR, device=DEVICE, is_vae=True)
vae_history = vae_trainer.fit(train_loader, val_loader, epochs=AE_EPOCHS, patience=AE_PATIENCE)

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Anomaly Detection — Training Curves', fontsize=14, fontweight='bold')

for name, history, color in [('Autoencoder', ae_history, '#2196F3'), ('VAE', vae_history, '#FF9800')]:
    epochs_range = range(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs_range, history['train_loss'], '--', color=color, alpha=0.5, label=f'{name} train')
    axes[0].plot(epochs_range, history['val_loss'], '-', color=color, label=f'{name} val')
    axes[1].plot(epochs_range, history['lr'], '-', color=color, label=name)

axes[0].set_title('Reconstruction Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Learning Rate'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('LR')
axes[1].set_yscale('log'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 9. Isolation Forest

In [ ]:
print('Training Isolation Forest...')

# Contamination = approximate attack ratio
contamination = float(y_train.mean())
contamination = min(max(contamination, 0.01), 0.45)
print(f'  Contamination ratio: {contamination:.3f}')

isolation_forest = IsolationForest(
    n_estimators=200,
    max_samples='auto',
    contamination=contamination,
    max_features=1.0,
    bootstrap=False,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
)

# Train on ALL data (Isolation Forest is unsupervised)
isolation_forest.fit(X_train)
print('  Isolation Forest trained.')

# Score: negative = anomaly, positive = normal
if_scores_train = -isolation_forest.decision_function(X_train)  # Negate so higher = more anomalous
if_scores_val = -isolation_forest.decision_function(X_val)
if_scores_test = -isolation_forest.decision_function(X_test)

print(f'  IF score range (train): [{if_scores_train.min():.4f}, {if_scores_train.max():.4f}]')
print(f'  IF score range (test):  [{if_scores_test.min():.4f}, {if_scores_test.max():.4f}]')

---
## 10. Compute Anomaly Scores for All Methods

In [ ]:
# Autoencoder scores
ae_scores_val, ae_labels_val = ae_trainer.compute_anomaly_scores(val_loader)
ae_scores_test, ae_labels_test = ae_trainer.compute_anomaly_scores(test_loader)

# VAE scores
vae_scores_val, vae_labels_val = vae_trainer.compute_anomaly_scores(val_loader)
vae_scores_test, vae_labels_test = vae_trainer.compute_anomaly_scores(test_loader)

print('Anomaly scores computed.')
print(f'  AE val range:  [{ae_scores_val.min():.4f}, {ae_scores_val.max():.4f}]')
print(f'  VAE val range: [{vae_scores_val.min():.4f}, {vae_scores_val.max():.4f}]')
print(f'  IF val range:  [{if_scores_val.min():.4f}, {if_scores_val.max():.4f}]')

## 11. Threshold Calibration

Find optimal threshold on validation set that maximizes F1 score.

In [ ]:
def find_optimal_threshold(scores, labels, method_name: str, n_thresholds: int = 200):
    """
    Find the threshold that maximizes F1 score on validation set.
    Returns: optimal threshold, best F1, AUC-ROC.
    """
    thresholds = np.linspace(np.percentile(scores, 1), np.percentile(scores, 99), n_thresholds)
    
    best_f1 = 0
    best_thresh = thresholds[0]
    f1_scores = []
    
    for thresh in thresholds:
        preds = (scores >= thresh).astype(int)
        f1 = f1_score(labels, preds, average='binary', zero_division=0)
        f1_scores.append(f1)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    
    try:
        auc = roc_auc_score(labels, scores)
    except ValueError:
        auc = 0.0
    
    print(f'  {method_name}: Best threshold={best_thresh:.4f}, F1={best_f1:.4f}, AUC={auc:.4f}')
    
    return best_thresh, best_f1, auc, thresholds, f1_scores

print('\nThreshold calibration (on validation set):')
ae_thresh, ae_f1, ae_auc, ae_threshs, ae_f1s = find_optimal_threshold(
    ae_scores_val, ae_labels_val, 'Autoencoder'
)
vae_thresh, vae_f1, vae_auc, vae_threshs, vae_f1s = find_optimal_threshold(
    vae_scores_val, vae_labels_val, 'VAE'
)
if_thresh, if_f1, if_auc, if_threshs, if_f1s = find_optimal_threshold(
    if_scores_val, y_val, 'Isolation Forest'
)

In [ ]:
# Plot threshold vs F1
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Threshold Calibration — F1 vs Threshold', fontsize=14, fontweight='bold')

for ax, name, threshs, f1s, best_t, best_f in [
    (axes[0], 'Autoencoder', ae_threshs, ae_f1s, ae_thresh, ae_f1),
    (axes[1], 'VAE', vae_threshs, vae_f1s, vae_thresh, vae_f1),
    (axes[2], 'Isolation Forest', if_threshs, if_f1s, if_thresh, if_f1),
]:
    ax.plot(threshs, f1s, 'b-', linewidth=2)
    ax.axvline(best_t, color='r', linestyle='--', label=f'Best: {best_t:.4f} (F1={best_f:.4f})')
    ax.set_title(name); ax.set_xlabel('Threshold'); ax.set_ylabel('F1 Score')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## 12. Ensemble Anomaly Scorer

In [ ]:
class EnsembleAnomalyScorer:
    """
    Combines multiple anomaly detection methods into an ensemble score.
    Normalizes each score to [0,1] then takes weighted average.
    """
    
    def __init__(self):
        self.scalers = {}
        self.weights = {}
        self.threshold = None
    
    def fit(self, scores_dict: dict, labels: np.ndarray, weights: dict = None):
        """
        Fit normalizers and find optimal ensemble threshold.
        scores_dict: {'method_name': scores_array}
        """
        # Default equal weights
        if weights is None:
            weights = {k: 1.0 / len(scores_dict) for k in scores_dict}
        self.weights = weights
        
        # Fit MinMax scalers for each method
        normalized_scores = {}
        for name, scores in scores_dict.items():
            scaler = MinMaxScaler()
            normalized = scaler.fit_transform(scores.reshape(-1, 1)).flatten()
            self.scalers[name] = scaler
            normalized_scores[name] = normalized
        
        # Compute weighted ensemble score
        ensemble_scores = np.zeros(len(labels))
        for name, normalized in normalized_scores.items():
            ensemble_scores += self.weights[name] * normalized
        
        # Find optimal threshold
        self.threshold, best_f1, auc, _, _ = find_optimal_threshold(
            ensemble_scores, labels, 'Ensemble'
        )
        
        return ensemble_scores
    
    def predict_scores(self, scores_dict: dict) -> np.ndarray:
        """Compute ensemble anomaly scores for new data."""
        ensemble_scores = np.zeros(len(next(iter(scores_dict.values()))))
        for name, scores in scores_dict.items():
            normalized = self.scalers[name].transform(scores.reshape(-1, 1)).flatten()
            normalized = np.clip(normalized, 0, 1)
            ensemble_scores += self.weights[name] * normalized
        return ensemble_scores
    
    def predict(self, scores_dict: dict) -> np.ndarray:
        """Predict anomaly labels (0=normal, 1=anomaly)."""
        ensemble_scores = self.predict_scores(scores_dict)
        return (ensemble_scores >= self.threshold).astype(int)

# Build ensemble
ensemble = EnsembleAnomalyScorer()

# Fit on validation set
val_scores_dict = {
    'ae': ae_scores_val,
    'vae': vae_scores_val,
    'if': if_scores_val,
}

# Weight by individual AUC performance
total_auc = ae_auc + vae_auc + if_auc
weights = {
    'ae': ae_auc / total_auc,
    'vae': vae_auc / total_auc,
    'if': if_auc / total_auc,
}
print(f'\nEnsemble weights: {weights}')

ensemble_val_scores = ensemble.fit(val_scores_dict, ae_labels_val, weights=weights)
print(f'Ensemble threshold: {ensemble.threshold:.4f}')

---
## 13. Test Set Evaluation

In [ ]:
def evaluate_anomaly_detector(scores, labels, threshold, method_name):
    """Evaluate anomaly detector on test set."""
    preds = (scores >= threshold).astype(int)
    
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='binary')
    try:
        auc = roc_auc_score(labels, scores)
        ap = average_precision_score(labels, scores)
    except ValueError:
        auc, ap = 0.0, 0.0
    
    print(f'\n{method_name}: Acc={acc:.4f} | F1={f1:.4f} | AUC={auc:.4f} | AP={ap:.4f}')
    print(classification_report(labels, preds, target_names=['Normal', 'Anomaly']))
    
    return {'method': method_name, 'accuracy': acc, 'f1': f1, 'auc': auc, 'ap': ap,
            'predictions': preds, 'scores': scores}

print('='*60)
print('  TEST SET EVALUATION')
print('='*60)

# Individual methods
ae_results = evaluate_anomaly_detector(ae_scores_test, ae_labels_test, ae_thresh, 'Autoencoder')
vae_results = evaluate_anomaly_detector(vae_scores_test, vae_labels_test, vae_thresh, 'VAE')
if_results = evaluate_anomaly_detector(if_scores_test, y_test, if_thresh, 'Isolation Forest')

# Ensemble
test_scores_dict = {
    'ae': ae_scores_test,
    'vae': vae_scores_test,
    'if': if_scores_test,
}
ensemble_test_scores = ensemble.predict_scores(test_scores_dict)
ensemble_results = evaluate_anomaly_detector(
    ensemble_test_scores, ae_labels_test, ensemble.threshold, 'Ensemble'
)

In [ ]:
# Visualization: ROC curves for all methods
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Anomaly Detection — Test Set Results', fontsize=14, fontweight='bold')

# ROC Curves
for name, scores, labels, color in [
    ('Autoencoder', ae_scores_test, ae_labels_test, '#2196F3'),
    ('VAE', vae_scores_test, vae_labels_test, '#FF9800'),
    ('Isolation Forest', if_scores_test, y_test, '#4CAF50'),
    ('Ensemble', ensemble_test_scores, ae_labels_test, '#E91E63'),
]:
    try:
        fpr, tpr, _ = roc_curve(labels, scores)
        auc = roc_auc_score(labels, scores)
        axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.4f})')
    except ValueError:
        pass

axes[0].plot([0,1], [0,1], 'r--', alpha=0.3)
axes[0].set_title('ROC Curves'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Score distributions (Ensemble)
normal_scores = ensemble_test_scores[ae_labels_test == 0]
anomaly_scores = ensemble_test_scores[ae_labels_test == 1]
axes[1].hist(normal_scores, bins=100, alpha=0.6, label='Normal', color='green', density=True)
axes[1].hist(anomaly_scores, bins=100, alpha=0.6, label='Anomaly', color='red', density=True)
axes[1].axvline(ensemble.threshold, color='black', linestyle='--', label=f'Threshold={ensemble.threshold:.3f}')
axes[1].set_title('Ensemble Score Distribution'); axes[1].set_xlabel('Score'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Confusion matrix (Ensemble)
cm = confusion_matrix(ae_labels_test, ensemble_results['predictions'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'])
axes[2].set_title('Ensemble Confusion Matrix'); axes[2].set_ylabel('True'); axes[2].set_xlabel('Predicted')

plt.tight_layout(); plt.show()

In [ ]:
# Comparison table
comparison_df = pd.DataFrame([
    {'Method': r['method'], 'Accuracy': r['accuracy'], 'F1': r['f1'], 'AUC': r['auc'], 'AP': r['ap']}
    for r in [ae_results, vae_results, if_results, ensemble_results]
]).sort_values('F1', ascending=False)

print('\n' + '='*60)
print('  ANOMALY DETECTION COMPARISON')
print('='*60)
print(comparison_df.to_string(index=False))

---
## 14. Save Models & Artifacts

In [ ]:
anomaly_dir = MODELS_DIR / 'anomaly_detection'
anomaly_dir.mkdir(parents=True, exist_ok=True)

# Save Autoencoder
torch.save({
    'model_state_dict': ae_model.state_dict(),
    'model_class': 'DeepAutoencoder',
    'n_features': N_FEATURES,
    'latent_dim': 16,
    'threshold': float(ae_thresh),
    'history': ae_history,
    'dataset': DATASET_NAME,
    'timestamp': datetime.now().isoformat(),
}, anomaly_dir / 'autoencoder.pth')
print(f'Saved autoencoder')

# Save VAE
torch.save({
    'model_state_dict': vae_model.state_dict(),
    'model_class': 'VAEAnomalyDetector',
    'n_features': N_FEATURES,
    'latent_dim': 16,
    'threshold': float(vae_thresh),
    'history': vae_history,
    'dataset': DATASET_NAME,
    'timestamp': datetime.now().isoformat(),
}, anomaly_dir / 'vae.pth')
print(f'Saved VAE')

# Save Isolation Forest
joblib.dump({
    'model': isolation_forest,
    'threshold': float(if_thresh),
    'dataset': DATASET_NAME,
    'timestamp': datetime.now().isoformat(),
}, anomaly_dir / 'isolation_forest.joblib')
print(f'Saved Isolation Forest')

# Save Ensemble config
joblib.dump({
    'scalers': ensemble.scalers,
    'weights': ensemble.weights,
    'threshold': ensemble.threshold,
    'methods': ['ae', 'vae', 'if'],
    'dataset': DATASET_NAME,
    'timestamp': datetime.now().isoformat(),
}, anomaly_dir / 'ensemble_config.joblib')
print(f'Saved ensemble config')

# Save comparison
comparison_df.to_csv(anomaly_dir / 'anomaly_comparison.csv', index=False)

print(f'\nAll anomaly detection models saved to {anomaly_dir}')

In [ ]:
# Inference helper
def load_anomaly_detector(models_dir: str, device: str = 'cpu'):
    """
    Load all anomaly detection models for inference.
    Returns: ae_model, vae_model, if_model, ensemble_config
    """
    models_path = Path(models_dir)
    
    # Load AE
    ae_ckpt = torch.load(models_path / 'autoencoder.pth', map_location=device, weights_only=False)
    ae = DeepAutoencoder(ae_ckpt['n_features'], ae_ckpt['latent_dim'])
    ae.load_state_dict(ae_ckpt['model_state_dict'])
    ae.to(device).eval()
    
    # Load VAE
    vae_ckpt = torch.load(models_path / 'vae.pth', map_location=device, weights_only=False)
    vae = VAEAnomalyDetector(vae_ckpt['n_features'], vae_ckpt['latent_dim'])
    vae.load_state_dict(vae_ckpt['model_state_dict'])
    vae.to(device).eval()
    
    # Load IF
    if_data = joblib.load(models_path / 'isolation_forest.joblib')
    
    # Load ensemble config
    ens_config = joblib.load(models_path / 'ensemble_config.joblib')
    
    return ae, vae, if_data['model'], ens_config

print('\n✓ Notebook 03 complete. Ready for Notebook 04 (Attack Classification).')

In [ ]:
gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
print('Memory freed.')